# Folder 01 / file 07 — Compare Adult tags only (after evaluate; parallel with XAI)

UCI Adult Census Income ([dataset](https://archive.ics.uci.edu/dataset/2/adult)): binary target `income_gt_50k` where **`>50K` = 1** and **`<=50K` = 0**. Fourteen census features; official split is `adult.data` (train) / `adult.test` (holdout).

Inlined replica of `src/n01_dev_train/n07_compare.py`. Run cells **in order** (local or Jobs). Catalog/schema/model/mode come from task env.

Compare Adult challenger vs champion on absolute-eval tags only (no aliases).


## 1 — Imports


In [ ]:
from mlflow.exceptions import MlflowException
from src.n00_shared.protocol import compare_decision
from src.n00_shared.runtime import Settings, assert_dev_ml_allowed, configure_mlflow, get_task_value, load_settings, mlflow_client, set_task_value


## 2 — `_tag_map`


In [ ]:
def _tag_map(client, name: str, version: str) -> dict:
    mv = client.get_model_version(name, version)
    tags = mv.tags or {}
    if isinstance(tags, dict):
        return tags
    return {t.key: t.value for t in tags}


## 3 — `_metric_from_tags`


In [ ]:
def _metric_from_tags(tags: dict, metric: str):
    raw = tags.get(metric)
    if raw is None or str(raw).strip() == "":
        return None
    return float(raw)


## 4 — `settings = load_settings()`


In [ ]:
settings = load_settings()


## 5 — `run()` step 1/3


In [ ]:
assert_dev_ml_allowed(settings)
configure_mlflow(settings)
client = mlflow_client()
version = get_task_value("evaluate", "model_version")
tags = _tag_map(client, settings.source_model_name, version)
challenger_metric = _metric_from_tags(tags, settings.compare_metric)
champion_exists = True
champion_metric = None


## 6 — `run()` step 2/3


In [ ]:
try:
    champ = client.get_model_version_by_alias(settings.source_model_name, "champion")
    champ_tags = _tag_map(client, settings.source_model_name, champ.version)
    champion_metric = _metric_from_tags(champ_tags, settings.compare_metric)
except MlflowException:
    champion_exists = False
except Exception:
    champion_exists = False


## 7 — `run()` step 3/3


In [ ]:
result, first_version = compare_decision(
    challenger_metric,
    champion_metric,
    settings.compare_higher_is_better,
    settings.compare_margin,
    champion_exists,
)
client.set_model_version_tag(settings.source_model_name, version, "compare_result", result)
if first_version:
    client.set_model_version_tag(settings.source_model_name, version, "first_version", "true")
set_task_value("compare_result", result)
set_task_value("model_version", version)
print(f"compare tags-only result={result} first_version={first_version}")
